# Embedding.py

This notebook is used to compute the vector embeddings for three variants of standardized LOINC lab names. The three variants covered are long common name, short name, and display name, for a total of ~276k embedded vectors. The notebook extracts the LOINC information from a tabular file in Azure Blob Storage, formats them as inputs to a `sentence-transformers` model, and then uses CUDA-GPU optimization to rapidly encode them into `pytorch Tensors`. These `Tensors` are then persisted into local memory using a simple `pickle` operation that pairs each vector with its associated LOINC code string.

Note that for optimal performance, a GPU-supported compute instance is **required**. CPU-operations get Tensor-processed at a rate of ~1 iteration/second, while GPU-operations can be processed as fast as 30 iterations/second. 

For larger models (e.g. those with many parameters), the notebook also supports mini-batching. This increases the time to compute embeddings but ensures the GPU isn't overloaded and no memory errors occur. For models of up to 8B parameters, mini-batching is not needed, and all vectors can be processed in memory. Any larger, and some mini-batching is required.

Make sure to run this notebook all the way through, including the last cell. The embeddings computation cell leaves a large file in Azure _local_ memory; to make these embeddings broadly available, such as for running e.g. the `performance.ipynb` notebook, the file must be moved to Azure Blob Storage. This is handled by the very last cell of this notebook, which also cleans up the local workspace to avoid lingering storage fees.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(
    vault_url=f"https://{key_vault}.vault.azure.net/",
    credential=credential,
)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = "workspaceblobstore"

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), so make sure the SNOINC extracts file is properly instantiated as an Azure resource before loading it.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}",
)

## Step 2: Load LOINC Code Strings

Using our file mount, we can read the tabular-formatted file of LOINC lab names for processing into a list format for our transformer model. Remember, the LOINC file needs to already be an Azure Data Asset before this will work. We decode the bytestrings into proper UTF-8, then combine the three names of interest to our work (long common name, short names, and display names) into a single list collection for the embedding model.

In [ ]:
SNOINC_CODES_FILE = "./loinc_lab_names_20251008.csv"

# Load up the validation set data
print("Extracting SNOINC data to form standardized names...")

long_common_names = []
short_names = []
display_names = []

with fs.open(SNOINC_CODES_FILE) as fp:
    # First line is a header giving the column names
    lines_seen = 0
    for line in fp:
        if lines_seen == 0:
            lines_seen += 1
            continue

        # Azure Blob Storage is bytes-based, so we need to apply utf decoding
        # before we can use string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            names = line_str.strip().split("|")
            # Skip lines that aren't real entries (formatting artifacts)
            if len(names) >= 4:
                long_common_names.append(names[2].strip())
                short_names.append(names[1].strip())
                display_names.append(names[3].strip())

# Filter out blanks created as a result of formatting/decoding artifacts
for name_list in [long_common_names, short_names, display_names]:
    name_list = [x for x in name_list if not x == ""]

# There should be ~270k loinc names
name_codes = long_common_names + short_names + display_names
assert len(name_codes) > 0

print(f"{len(name_codes)} name codes loaded")

## Step 3: Verify GPU Operation

This step may not look like much, but a properly functioning GPU is _imperative_ to the processing capabilities of this script. Pytorch _should_ automatically be able to detect whether the local compute instance is GPU-capable, but when dealing with cloud infrastructure it's best to be sure. We don't even want to try running this if `cuda` isn't accessible.

In [ ]:
import torch

assert torch.cuda.is_available()

## Step 4: Instantiate Language Model

Here, we'll plug in the name (defined on the hugging face page for a particular model) of the model we want to embed. When we load the model's properties, we will _explicitly_ toggle it into CUDA-based GPU mode, since we checked above the compute instance can support it.

In [ ]:
from sentence_transformers import SentenceTransformer

# Instantiate the language model
MODEL_NAME = "intfloat/e5-large-v2"

model = SentenceTransformer(MODEL_NAME, device="cuda")
n_params = sum(p.numel() for p in model.parameters())
print(f"Model uses {n_params} parameters")

## Step 5: Compute Embeddings

This is the bulk of the work of this notebook. Embeddings get computed here, either one-shot or via mini-batching.

For most models the TtC team is working with, it is likely sufficient to encode all values at once, without needing to rely on mini-batching. We tested models up to 8B parameters in size and encountered no issues during encoding or storage. In this case, set `USE_INCREMENTAL_MINI_BATCHING` to `False`. We also found that a constant `BATCH_SIZE` of 32 was highly performant.

For larger models, or any time the notebook struggles to keep all encodings in memory (due to their size or the number of floating-point operations), you can switch the encoder to mini-batching mode by setting the appropriate boolean to `True`. If mini-batching is used, the `CHUNK_SIZE` parameter defines the size of the mini-batched segment (i.e. how many LOINC code strings are processed before the results are persisted back to disk). We recommend a default value of 8192, but this value can and should be adjusted based on how much the GPU can take before crashing. A higher mini-batching value will enable faster processing, but will put more load on the GPU virtual RAM per batch. If the vector embeddings of a mini-batch can't fit in this VRAM, then the compute instance will crash, and the mini-batch size should be lowered. Just make sure to balance this batch size against the additional time required to repeatedly save and load from disk.

**Troubleshooting Tip:** When handling the large models, or if you have run this notebook several times in sequence, it's possible the compute instance will run out of cache memory. Huggingface downloads the entire backend of every model we instantiate with `sentence-transformers`, which is convenient for our processing but is cumbersome for storage. If you run this cell and are told you're out of memory or the cache is too full, follow these simple steps to clear it:

* Click the three dots next to the compute dropdown and Open Terminal
* Navigate to the compute instance's home directory with `cd ~`
* Run `ls -al` and make sure you can see the directory `.cache` listed. If it's not there, you're not in the right home directory.
* `cd` into `.cache/`. If you run `ls -al`, you should see a folder named `huggingface` or `.huggingface` (depending on settings--which one is present doesn't matter, they're both cache folders).
* Run `rm -rf huggingface/` and wait for the operation to complete. Exit the terminal window (you'll get a pop-up that this will terminate running processes, that's fine). Run the cell again and you're good to go.


In [ ]:
import pickle

# This value is always used, regardless of mini-batching mode. Higher batch
# sizes allow the encoder to process more code strings in parallel, but will
# slow down the overall pace of iterations/second. 32 is a very reasonable
# default.
BATCH_SIZE = 32

# This value is only used in mini-batching mode
CHUNK_SIZE = 8192

DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]
EMBEDDING_FILE = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"

USE_INCREMENTAL_MINI_BATCHING = False

print("Performing embedding, this might take a while...")
if USE_INCREMENTAL_MINI_BATCHING:
    # We'll iterate in chunk-sized intervals to not overload the GPU
    start = 0
    while start < len(name_codes):
        end = min(start + CHUNK_SIZE, len(name_codes))
        mini_batch = name_codes[start : min(end, len(name_codes))]

        print("Mini-batching from", start, "to", end - 1)

        # Encoding as a Tensor keeps the result in GPU after calculating it,
        # which allows us to "stack" it on top of the previously computed
        # embeddings in the next step
        batch_embeddings: torch.Tensor = model.encode(
            mini_batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
        )

        try:
            # No direct appending to pickle file, so read it, extend it,
            # overwrite it. Torch.cat is super efficient so this isn't a
            # problem to re-allocate the objects.
            with open(EMBEDDING_FILE, "rb") as fp:
                cache_data = pickle.load(fp)
            file_codes: list = cache_data["codes"]
            saved_embeddings: torch.Tensor = cache_data["embeddings"]
            file_codes.extend(mini_batch)
            extended_embeddings = torch.cat((saved_embeddings, batch_embeddings), dim=0)

            with open(EMBEDDING_FILE, "wb") as fp:
                pickle.dump({"codes": file_codes, "embeddings": extended_embeddings}, fp)

        except FileNotFoundError:
            # File doesn't exist, so just create it
            with open(EMBEDDING_FILE, "wb") as fp:
                pickle.dump({"codes": mini_batch, "embeddings": batch_embeddings}, fp)

        start += CHUNK_SIZE

else:
    corpus_embeddings = model.encode(
        name_codes,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_tensor=True,
    )
    with open(EMBEDDING_FILE, "wb") as fp:
        pickle.dump({"codes": name_codes, "embeddings": corpus_embeddings}, fp)

## Step 6: Cleanup File Locations

With the emebddings computed, we simply move the pickled file into Azure's Blob Storage using our file mount, and then remove the local copy. When this cell completes, if you refresh the notebook pane on the left (the clockwise circular arrow above the dropdown folders), you should see _no_ embedding file in your local workspace. Similarly, in Blob Storage, in the explorer pane, if you refresh there, you _should_ see the embedding file appear.

In [ ]:
import os

fs.put(EMBEDDING_FILE, "/embeddings/")
os.remove(EMBEDDING_FILE)